# Does the gate pick better frames for a VLM? EgoEverything, 12 test videos, Qwen2.5-VL-3B

[EgoEverything](https://huggingface.co/datasets/roxqtang/EgoEverything) (arXiv 2604.08342)
asks 5-option questions about what an AR-glasses wearer saw, built from real gaze traces on
Aria Everyday Activities (AEA), the same videos the gate was trained and tested on. Each
question names a **keyframe time** where the answer is visible.

**The test:** at the **same number of frames**, does the gate's choice of frames give the VLM
better answers than spreading them evenly?

| condition | frames given to the VLM (from the same 1 fps frames as the labels) |
|---|---|
| **D: full** | policy D (send when the view settles + on each REFIXATION), driven by the eye + IMU + cos model |
| D: eye + IMU | the same, eye + IMU model |
| D: old gate | the same, driven by the old gaze-only gate |
| D: ground truth | the same, on the DINOv2 labels: the policy's ceiling |
| **uniform (same budget)** | as many frames per video as D: full, evenly spaced: **the key baseline** |
| random (same budget) | as many, at random |
| blind (no frames) | question and options only: the floor (chance is 20%) |

Only the 12 held-out **test** videos are used, so no gate model has seen them. That gives
roughly a couple of hundred questions: a pilot. Expect wide intervals.

## Run order

1. Clone, mount, install
2. **AEA links JSON**: upload the fresh one the first time; it replaces the older copies on
   Drive, and later runs reuse it
3. EgoEverything questions on the 12 test videos
4. Which frames each condition sends
5. Download the 12 preview MP4s once, extract 1 fps frames, cache them on Drive
6. Check alignment (keyframe boxes on our frames) and **target coverage**: does each
   condition send a frame near the moment each question is about? No VLM needed
7. Load Qwen2.5-VL-3B, smoke test
8. Score every condition (resumable)
9. Results

**Runtime:** a T4 works (fp16). An L4/A100 is faster and uses bf16. Section 8 is the long
one: about 1–2 h on a T4 for all conditions. It writes each answer to Drive as it goes, so a
disconnect costs nothing; just rerun.

## 1. Clone, mount, install

In [ ]:
import os, sys, json, subprocess

REPO, BRANCH = '/content/GazeVLM', 'eyehead-gate'     # 'main' once merged
URL = 'https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git'
if os.path.exists(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
assert os.path.exists('vlmeval/qwen_mcq.py'), f'vlmeval/ is not on branch {BRANCH!r}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'qwen-vl-utils', 'huggingface_hub'],
               check=True)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/GazeVLM'
FOLDS     = os.path.join(DRIVE_DIR, 'folds')
RUNS      = os.path.join(DRIVE_DIR, 'runs')
VLM_DIR   = os.path.join(DRIVE_DIR, 'vlm', 'egoeverything')
FRAMES_TAR = os.path.join(VLM_DIR, 'frames_tar')          # one .tar per video, reused later
LOCAL_FRAMES = '/content/frames_1fps'
os.makedirs(VLM_DIR, exist_ok=True)

PREDS = {'full':      os.path.join(RUNS, 'eyehead', 'test_preds.csv'),
         'eye + IMU': os.path.join(RUNS, 'eyehead_eye_head', 'test_preds.csv'),
         'old gate':  os.path.join(DRIVE_DIR, 'release', 'gate_v1', 'test_preds.csv')}
for k, p in PREDS.items():
    print(f'{k:<10} {"found" if os.path.exists(p) else "MISSING"}  {p}')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 2. The AEA download-links JSON

The signed links expire after ~14 days. This cell checks the copy on Drive:

- **missing or expired** → it asks you to upload the fresh `aea_download_urls.json`, checks
  its links work, and **replaces the older copies** on Drive (`MyDrive/GazeVLM/` and, if
  present, `MyDrive/aea/`, so every notebook picks up the fresh one)
- **still valid** → nothing to do

Set `FORCE_UPLOAD = True` to replace it anyway.

In [ ]:
from vlmeval.frames import links_fresh, install_links

FORCE_UPLOAD = False
LINKS = os.path.join(DRIVE_DIR, 'aea_download_urls.json')
OLD_COPIES = [LINKS] + [p for p in ['/content/drive/MyDrive/aea/aea_download_urls.json']
                        if os.path.exists(p)]

ok, msg = links_fresh(LINKS) if os.path.exists(LINKS) else (False, 'not on Drive')
print('Drive copy:', msg)
if FORCE_UPLOAD or not ok:
    from google.colab import files
    print('upload the fresh aea_download_urls.json ...')
    up = files.upload()
    name = list(up.keys())[0]
    ok, msg = links_fresh(name)
    print('uploaded file:', msg)
    assert ok, 'the uploaded JSON does not work either -- download a fresh one from projectaria.com/datasets/aea'
    install_links(name, OLD_COPIES)
    os.remove(name)                      # the copies live on Drive; keep the repo clean
    for p in OLD_COPIES:
        print('   replaced', p)
links = json.load(open(LINKS))
print(f"using {LINKS}: {len(links['sequences'])} sequences")
print('(the eyehead notebook\'s IMU calibration, section 3, can now download its VRS too)')

## 3. EgoEverything on the 12 test videos

Loads the questions from Hugging Face and keeps the ones whose video is a gate **test**
video. Also prints how many questions fall on all 70 gate videos, for scaling up later.

In [ ]:
from vlmeval import egoev

qs_all = egoev.load_questions()
aea = qs_all[qs_all.video.str.startswith('loc')]
clips, test = egoev.gate_clips(FOLDS, test_only=True)
test_videos = [c['sequence'] for c in clips]
n_frames = {c['sequence']: len(c['frame_no']) for c in clips}

all70 = set(pd.read_csv(os.path.join(FOLDS, 'train_val_fold1.csv'), usecols=['sequence']).sequence) | set(test_videos)
print(f'EgoEverything: {len(qs_all):,} questions, {len(aea):,} on AEA '
      f'({aea.video.nunique()} AEA videos)')
print(f'   on any of the 70 gate videos: {aea.video.isin(all70).sum():,} questions, '
      f'{aea[aea.video.isin(all70)].video.nunique()} videos')

qs = aea[aea.video.isin(test_videos)].reset_index(drop=True)
print(f'   on the 12 TEST videos:        {len(qs):,} questions, {qs.video.nunique()} videos\n')
assert len(qs), 'no EgoEverything question falls on a test video'

per = qs.groupby('video').agg(questions=('qid', 'size'), t0_min=('t0', 'min'),
                              t0_max=('t0', 'max'), median_dist2gaze=('dist2gaze', 'median'))
per['video_s'] = [n_frames[v] for v in per.index]
display(per.round(1))
print('answer letters:', qs.answer.value_counts().sort_index().to_dict(), '(balanced = no letter bias)')
print('example:', qs.question.iloc[0], qs.options.iloc[0], '->', qs.answer.iloc[0])

## 4. Which frames each condition sends

The uniform and random conditions get, per video, exactly as many frames as **D: full**.
The selection is saved to Drive.

In [ ]:
sel = egoev.select_frames(clips, test, PREDS, main='full', seed=0)
json.dump(sel, open(os.path.join(VLM_DIR, 'selection.json'), 'w'))
display(egoev.budget_table(sel, n_frames).round(2))

## 5. Frames: download once, cache on Drive

Downloads the 12 preview MP4s (~165 MB each, no VRS), keeps every 1 s frame with the
pipeline's own rule, resizes to 448 px and stores one `.tar` per video on Drive. Later runs
untar from Drive and skip the download. The frame count must match the label sequence.

In [ ]:
from vlmeval.frames import ensure_frames, frame_path

SIZE = 448
metas = {}
for v in test_videos:
    metas[v] = ensure_frames(v, links, LOCAL_FRAMES, FRAMES_TAR, size=SIZE)
    m = metas[v]
    flag = 'OK' if m['n'] >= n_frames[v] else f'!! only {m["n"]} frames, labels need {n_frames[v]}'
    print(f"   {v:<26} {m['n']:4d} frames  ({m['native_fps']:.1f} fps source, step {m['step']}, "
          f"orig {m['orig_w']}x{m['orig_h']})  {flag}")

## 6. Alignment check and target coverage

**Alignment:** each panel is the frame nearest a question's keyframe time, with that
question's box drawn on it. If the boxes sit on the named objects, our frame numbers, the
question timestamps and the box coordinates all line up. If they are consistently off,
stop here: the coverage and VLM numbers below would be wrong.

**Target coverage:** for each question, the distance from its keyframe to the nearest frame
a condition sends. A policy that rarely sends a frame near the moment a question is about
cannot help the VLM, however good the VLM is. This needs no GPU.

In [ ]:
import matplotlib.patches as mpatches

def frame_time(v, k):
    return metas[v]['ts'][k] if k < metas[v]['n'] else k

show = qs[qs.bbox.notna() & qs.t0.notna()].sample(min(6, len(qs)), random_state=0)
fig, ax = plt.subplots(1, len(show), figsize=(3.4 * len(show), 3.8))
for a_, q in zip(np.atleast_1d(ax), show.itertuples()):
    m = metas[q.video]
    k = int(np.argmin(np.abs(np.array(m['ts']) - q.t0)))
    img = plt.imread(frame_path(m, k))
    s = img.shape[1] / m['orig_w']
    x1, y1, x2, y2 = [b * s for b in q.bbox]
    a_.imshow(img)
    a_.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='yellow', lw=2))
    a_.set_title(f"{q.object}\nt0={q.t0:.1f}s  frame {k}", fontsize=8); a_.axis('off')
plt.tight_layout(); plt.show()

cov = egoev.coverage(qs, sel, frame_time)
ctab = cov.groupby('condition', sort=False).agg(
    questions=('qid', 'size'), median_dist_s=('dist', 'median'),
    **{f'within_{t}': (f'hit_{t}', 'mean') for t in ('1s', '2s', '5s')})
for c in ('within_1s', 'within_2s', 'within_5s'):
    ctab[c] *= 100
display(ctab.round(1))
print('within_1s = % of questions whose keyframe has a sent frame no more than 1 s away.')

## 7. Load Qwen2.5-VL-3B and smoke test

`MAX_PIXELS = 448*448` gives about 256 visual tokens per frame. Lower it (e.g. `336*336`) if
the T4 runs out of memory or is too slow. Keep it the same for every condition.

In [ ]:
from vlmeval import qwen_mcq

MODEL_ID, MAX_PIXELS = 'Qwen/Qwen2.5-VL-3B-Instruct', 448 * 448
model, proc, letter_ids, dtype = qwen_mcq.load_model(MODEL_ID)
print(MODEL_ID, dtype, f'{torch.cuda.memory_allocated()/1e9:.1f} GB on GPU')

import time
q = qs.iloc[0]
frames = [(frame_path(metas[q.video], k), metas[q.video]['ts'][k])
          for k in sel['D: full'][q.video]]
t0 = time.time()
p, ntok = qwen_mcq.score(model, proc, letter_ids,
                         qwen_mcq.build_messages(q.question, q.options, frames, MAX_PIXELS))
print(f'{len(frames)} frames, {ntok} prompt tokens, {time.time()-t0:.1f} s')
print('Q:', q.question)
for L, o, pi in zip('ABCDE', q.options, p):
    print(f"   {L}. {o:<60} p={pi:.3f}{'   <- answer' if L == q.answer else ''}")
assert np.isfinite(p).all(), ('NaN logits: fp16 overflow on this GPU. Switch the runtime to an '
                              'L4/A100 (bf16) or lower MAX_PIXELS.')

## 8. Score every condition

Resumable: answers are appended to a JSONL file on Drive, and a rerun skips what is done.
The key conditions run first, so stopping early still leaves the main comparison.

In [ ]:
CONDITIONS = ['D: full', 'uniform (same budget)', 'blind (no frames)', 'D: ground truth',
              'random (same budget)', 'D: eye + IMU', 'D: old gate']
CONDITIONS = [c for c in CONDITIONS if c in sel]
OUT = os.path.join(VLM_DIR, f"{MODEL_ID.split('/')[-1]}_px{int(MAX_PIXELS**0.5)}.jsonl")
qwen_mcq.run(model, proc, letter_ids, qs, sel, metas, OUT, CONDITIONS, MAX_PIXELS, tag=MODEL_ID)

## 9. Results

- **Accuracy per condition**, with a 95% interval from resampling the test videos.
- **Paired comparisons** on the same questions: D: full against uniform, random and blind,
  and the ceiling against uniform. `a_only_right` / `b_only_right` count the questions only
  one condition got right; `p_mcnemar` tests whether that split is more lopsided than chance.
- **Accuracy by target coverage:** questions where a condition did or did not send a frame
  within 2 s of the keyframe.

In [ ]:
res = egoev.load_results(OUT)
bad = res[res['error'].notna()] if 'error' in res else res.iloc[:0]
if len(bad):
    print(f'{len(bad)} questions not scored (oom/nan), left out below:')
    display(bad.groupby(['condition', 'error']).size())
res = res[res['error'].isna()].copy() if 'error' in res else res.copy()
res['correct'] = res['correct'].astype(int)

acc = egoev.accuracy_table(res)
display(acc.round(2))

pairs = [('D: full', 'uniform (same budget)'), ('D: full', 'random (same budget)'),
         ('D: full', 'blind (no frames)'), ('D: ground truth', 'uniform (same budget)'),
         ('D: full', 'D: old gate'), ('D: eye + IMU', 'uniform (same budget)')]
pt = pd.DataFrame([egoev.paired_test(res, a, b) for a, b in pairs
                   if a in set(res.condition) and b in set(res.condition)])
print('paired, same questions (diff = a - b, accuracy points):')
display(pt.round(3))

m = res.merge(cov[['condition', 'qid', 'hit_2s']], on=['condition', 'qid'], how='left')
by_cov = (m[m.condition != 'blind (no frames)']
          .groupby(['condition', 'hit_2s']).correct.agg(['mean', 'size']))
by_cov['mean'] *= 100
print('accuracy (%) when a frame within 2 s of the keyframe was / was not sent:')
display(by_cov.round(1).unstack())

fig, ax = plt.subplots(figsize=(9, 4.5))
order = acc.sort_values('accuracy').index
ax.barh(order, acc.loc[order, 'accuracy'],
        xerr=[acc.loc[order, 'accuracy'] - acc.loc[order, 'ci_low'],
              acc.loc[order, 'ci_high'] - acc.loc[order, 'accuracy']], color='tab:blue', alpha=.8)
ax.axvline(20, ls=':', c='k', lw=1, label='chance (20%)')
ax.set_xlabel('accuracy (%)'); ax.set_title(f'{MODEL_ID}, {len(qs)} questions, 12 test videos')
ax.legend(); plt.tight_layout(); plt.show()

---

## Reading it

| look at | meaning |
|---|---|
| **D: full vs uniform (same budget)** | the main result: does choosing frames by eye + head motion beat even spacing at equal cost? |
| D: ground truth vs uniform | the most this policy could gain with perfect labels |
| blind | how much the questions can be answered from text alone |
| coverage `within_2s` | whether a condition even sends the moment a question is about; if D does not cover more than uniform, a VLM gain is unlikely |
| accuracy by coverage | whether sending the keyframe actually helps this VLM |
| the intervals | with ~12 videos they are wide; treat this as a pilot, then scale to all 70 videos (held-out predictions per fold) |

## Caveats

- Qwen2.5-VL-3B is small; the paper reports 49% for VideoLLaMA3-7B with all frames at full
  resolution. Absolute numbers will be lower; the comparison between conditions is what
  matters.
- Frames are 448 px for every condition, so resolution does not differ between conditions.
- EgoEverything questions are about moments at least 3 minutes before the question; AEA
  videos are 1–8 minutes long, so many targets sit early in the video.